# 07. Split-Apply-Combine Groupby Operations (5+ Years Interview Guide)
Exhaustive revision guide to df.groupby(), multi-metric .agg(), named aggregations, .transform(), and .filter() on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Grouping**: Dedicated cell for `df.groupby()`.
- **Multiple Aggregations**: Dedicated cell for `.agg()` with multiple metrics.
- **Named Aggregations**: Dedicated cell for named aggregation tuples (`total_amt=('transaction_amount', 'sum')`).
- **Group Transformation**: Dedicated cell for `.transform()`.
- **Group Filtering**: Dedicated cell for `.filter()`.

This interactive revision guide uses `data/raw_transactions.csv` for all real-world code examples.

In [ ]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

### Grouping with `df.groupby()`
**Explanation**: Partitions transactions by region and card_type.

**Syntax**: `df.groupby(['region', 'card_type'])`

In [ ]:
grouped_region = df.groupby('region')

### Multi-Metric Aggregation with `.agg()`
**Explanation**: Computes total spend, average spend, and fraud rate per region.

**Syntax**: `df.groupby('region').agg({'transaction_amount': ['sum', 'mean'], 'is_fraud': 'mean'})`

In [ ]:
regional_summary = df.groupby('region').agg({
    'transaction_amount': ['sum', 'mean', 'count'],
    'is_fraud': 'mean'
})
print('Regional Multi-Metric Summary:\n', regional_summary)

### Named Aggregations Syntax
**Explanation**: Generates clean single-level column names for aggregated transaction metrics.

**Syntax**: `df.groupby('region').agg(total_spend=('transaction_amount', 'sum'), avg_spend=('transaction_amount', 'mean'), fraud_rate=('is_fraud', 'mean'))`

In [ ]:
clean_named_agg = df.groupby('region').agg(
    total_spend=('transaction_amount', 'sum'),
    avg_spend=('transaction_amount', 'mean'),
    fraud_rate=('is_fraud', 'mean'),
    tx_count=('transaction_id', 'count')
)
print('Clean Named Aggregations Table:\n', clean_named_agg.round(3))

### Dimension-Preserving Group `.transform()`
**Explanation**: Calculates each customer's mean transaction amount and computes individual transaction deviation z-scores.

**Syntax**: `df.groupby('customer_id')['transaction_amount'].transform('mean')`

In [ ]:
df_sub = df.dropna(subset=['transaction_amount']).head(100).copy()
df_sub['cust_avg_amt'] = df_sub.groupby('customer_id')['transaction_amount'].transform('mean')
df_sub['amt_ratio'] = df_sub['transaction_amount'] / df_sub['cust_avg_amt']
print(df_sub[['transaction_id', 'customer_id', 'transaction_amount', 'cust_avg_amt', 'amt_ratio']].head(5))

### Group Filtering with `.filter()`
**Explanation**: Extracts all transactions from high-volume customer accounts (customers with > 5 transactions).

**Syntax**: `df.groupby('customer_id').filter(lambda g: len(g) >= 5)`

In [ ]:
high_vol_cust = df.groupby('customer_id').filter(lambda g: len(g) >= 5)
print(f'Transactions from High-Volume Customers: {len(high_vol_cust)}')
print(high_vol_cust[['transaction_id', 'customer_id', 'transaction_amount']].head(3))

## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Identifying Outlier Transactions with Group Z-Scores
**Explanation**: Flag transactions that are 3 standard deviations above the customer's average spend.

**Syntax**: `(df['amt'] - df.groupby('cust')['amt'].transform('mean')) / df.groupby('cust')['amt'].transform('std') > 3`

In [ ]:
cust_means = df.groupby('customer_id')['transaction_amount'].transform('mean')
cust_stds = df.groupby('customer_id')['transaction_amount'].transform('std').fillna(1.0)
df_flagged = df.assign(z_score=(df['transaction_amount'] - cust_means) / cust_stds)
outliers = df_flagged[df_flagged['z_score'] > 2.5]
print(f'Found {len(outliers)} statistical outlier transactions!')
print(outliers[['transaction_id', 'customer_id', 'transaction_amount', 'z_score']].head(3))